In [1]:
# creates graphics in a new windows.
%matplotlib qt

import numpy as np 

import hyperspy.api as hs
import lumispy as lum

from h5py import File

# plot multiple inline figures side-by-side horizontally 
hs.preferences.Plot.widget_plot_style = 'horizontal'

C:\Users\jonny\anaconda3\envs\HyperSpy\Lib\site-packages\rsciio\utils\tools.py:93: VisibleDeprecationWarning: append2pathname has been moved to `rsciio.utils.path` and will be removed from `rsciio.utils.tools` in version 1.0.
  warnings.warn(message, VisibleDeprecationWarning)
C:\Users\jonny\anaconda3\envs\HyperSpy\Lib\site-packages\rsciio\utils\tools.py:93: VisibleDeprecationWarning: incremental_filename has been moved to `rsciio.utils.path` and will be removed from `rsciio.utils.tools` in version 1.0.
  warnings.warn(message, VisibleDeprecationWarning)
C:\Users\jonny\anaconda3\envs\HyperSpy\Lib\site-packages\rsciio\utils\tools.py:93: VisibleDeprecationWarning: get_file_handle has been moved to `rsciio.utils.file` and will be removed from `rsciio.utils.tools` in version 1.0.
  warnings.warn(message, VisibleDeprecationWarning)
C:\Users\jonny\anaconda3\envs\HyperSpy\Lib\site-packages\rsciio\utils\rgb_tools.py:46: VisibleDeprecationWarning: The module `rsciio.utils.rgb_tools` has been re

In [2]:
def load_map(file_path, dataset_path):
    r''' Loads numpy array from .h5 delmic file.
    file path - location of .h5
    dataset_path - location of 
    '''
    with File(file_path, 'r') as f:
        data = f[dataset_path][()]  
    return data

In [3]:
# --- 1. SETUP INTERACTIVE BACKEND ---
# If a separate window doesn't open, try changing 'qt' to 'widget' (requires ipympl)
try:
    %matplotlib qt
    print("✅ Switched to Interactive (Qt) Backend.")
except:
    print("⚠️ Could not switch backend. Ensure you are in Jupyter and have PyQt5 installed.")

import warnings
import matplotlib.pyplot as plt
import numpy as np
import hyperspy.api as hs
import warnings
import numpy as np
import pandas as pd
import hyperspy.api as hs
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
import os
from datetime import datetime

# --- 2. SILENCE WARNINGS (Updated for new Numpy) ---
# We filter general UserWarnings from rsciio, avoiding the 'VisibleDeprecationWarning' crash
warnings.filterwarnings("ignore", module="rsciio")

# --- 3. LOAD DATA ---
path = 'data/240326TE_S2_CL_5kV_1p6nA_1200_8K_004.h5'
raw_data = load_map(path, 'Acquisition2/ImageData/Image')
data = raw_data.squeeze()
excel_path = 'literature_peaks.xlsx'

✅ Switched to Interactive (Qt) Backend.


In [4]:
# --- 4. FIX DATA SHAPE (The "Smart" Transpose) ---
# We need the Energy dimension (Length ~2048) to be FIRST (Index 0)
# We need the Spatial dimensions (Length ~50-100) to be SECOND and THIRD
if data.shape[-1] > data.shape[0]: 
    # If the last dimension is the biggest (e.g. 2048), it's the spectrum. Move it to front.
    print(f"Transposing data from {data.shape}...")
    data = data.transpose(2, 0, 1)

# Flip so Energy increases (low -> high)
data = np.flip(data, axis=0)

# --- 5. CREATE SIGNAL & CALIBRATE ---
# Signal2D = "Image Stack". The first dimension automatically becomes the slider.
signal = hs.signals.Signal2D(data)

# Load energy axis
hc = 1239.84193
wavelengths = load_map(path, 'Acquisition2/ImageData/DimensionScaleC') * 1e9
energy_axis = np.sort(hc / wavelengths)

# Assign Slider (Navigation) Axis
# We trim to min_len to prevent crash if axis is 1 pixel off from data
min_len = min(len(energy_axis), data.shape[0])
signal.axes_manager.navigation_axes[0].axis = energy_axis[:min_len]
signal.axes_manager.navigation_axes[0].name = "Energy"
signal.axes_manager.navigation_axes[0].units = "eV"

# Assign Image (Signal) Axes
signal.axes_manager.signal_axes[0].scale = 0.009
signal.axes_manager.signal_axes[0].units = "μm"
signal.axes_manager.signal_axes[1].scale = 0.009
signal.axes_manager.signal_axes[1].units = "μm"

# --- 6. PLOT ---
print(f"Plotting data with shape: {signal.data.shape}")
print("Check your taskbar for a new window!")
signal.plot()

Plotting data with shape: (2048, 50, 50)
Check your taskbar for a new window!


In [5]:
hc = 1239.84193
tolerance = 0.05 

base_name = os.path.splitext(os.path.basename(path))[0]
timestamp = datetime.now().strftime("%m%y%d_%H%M%S")
output_csv_name = f"{base_name}_PeakSearch_{timestamp}.csv"

# --- 2. DATA LOADING & PREP ---
raw_data = load_map(path, 'Acquisition2/ImageData/Image')
data = raw_data.squeeze()
wavelengths = load_map(path, 'Acquisition2/ImageData/DimensionScaleC') * 1e9

if data.shape[-1] > data.shape[0]: 
    data = data.transpose(2, 0, 1)

energy_axis = np.sort(hc / wavelengths)
avg_data_raw = np.mean(data, axis=(1, 2)).flatten()
spec_data_1d = np.flip(avg_data_raw) 

s_average = hs.signals.Signal1D(spec_data_1d)
s_average.axes_manager[0].axis = energy_axis[:len(spec_data_1d)]
s_average.axes_manager[0].name = "Energy"
s_average.axes_manager[0].units = "eV"

# --- 3. FITTING (Background + Peaks) ---
m = s_average.create_model()
bg = hs.model.components1D.Polynomial(order=1) 
bg.name = "Background"
m.append(bg)

# Background Fit
try:
    m.set_signal_range(energy_axis.min(), energy_axis.min() + 0.5)
    m.fit()
    m.reset_signal_range() 
except: pass
bg.set_parameters_not_free()

# Peak Fit
indices, _ = find_peaks(spec_data_1d, prominence=spec_data_1d.max()*0.015, distance=15)
found_energies = [s_average.axes_manager[0].index2value(i) for i in indices]

for i, pos in enumerate(found_energies):
    if energy_axis.min() < pos < energy_axis.max():
        g = hs.model.components1D.Gaussian()
        g.name = f"Peak_{i}"
        g.centre.value = pos
        g.centre.bmin, g.centre.bmax = pos - 0.15, pos + 0.15
        m.append(g)

print(f"Fitting {len(found_energies)} peaks...")
m.fit(bounded=True)

# --- 4. CSV OUTPUT ---
try:
    df_lit = pd.read_excel(excel_path)
    fitted_centers = [comp.centre.value for comp in m if "Peak" in comp.name]
    final_results = []
    
    for _, row in df_lit.iterrows():
        lit_ev = row['Peak_eV']
        if not (energy_axis.min() <= lit_ev <= energy_axis.max()): continue
            
        if fitted_centers:
            diffs = np.abs(np.array(fitted_centers) - lit_ev)
            best_idx = np.argmin(diffs)
            
            if diffs[best_idx] <= tolerance:
                exp_ev = fitted_centers[best_idx]
                shift = (exp_ev - lit_ev) * 1000
                final_results.append({'Assignment': row['Assignment'], 'Status': 'MATCH', 'Exp_eV': exp_ev, 'Shift_meV': shift})
            else:
                final_results.append({'Assignment': row['Assignment'], 'Status': 'NOT FOUND'})
        else:
             final_results.append({'Assignment': row['Assignment'], 'Status': 'NO PEAKS FITTED'})

    pd.DataFrame(final_results).to_csv(output_csv_name, index=False)
    print(f"✅ CSV saved to: {output_csv_name}")
    
except Exception as e:
    print(f"Literature check failed: {e}")


# --- 5. WINDOW 1: MODEL + DATA ---
print("Displaying Window 1: Model Fit...")
m.plot(plot_components=True)
plt.title(f"Model Fit: {base_name}")
plt.xlabel("Energy (eV)")
plt.ylabel("Intensity (a.u.)")
plt.show()

# --- 6. WINDOW 2: LITERATURE LINES + DATA ---
print("Displaying Window 2: Literature Map...")
plt.figure("Literature Comparison Map", figsize=(10, 6))

# Plot the Experimental Data
plt.plot(energy_axis, spec_data_1d, color='black', linewidth=1.5, label='Experimental Data')

# Overlay Literature Lines
try:
    visible_lit = df_lit[(df_lit['Peak_eV'] >= energy_axis.min()) & 
                         (df_lit['Peak_eV'] <= energy_axis.max())]
    
    y_min, y_max = plt.ylim() # Get limits from data plot
    
    for _, row in visible_lit.iterrows():
        ev = row['Peak_eV']
        label = row['Assignment']
        
        plt.axvline(x=ev, color='red', linestyle='-', alpha=0.5, linewidth=1.5)
        plt.text(ev, y_max * 0.85, f"  {label}", 
                 rotation=90, color='red', fontsize=10, verticalalignment='center', fontweight='bold')

except Exception as e:
    print(f"Could not plot literature lines: {e}")

plt.title(f"Literature Reference Map: {base_name}")
plt.xlabel("Energy (eV)")
plt.ylabel("Intensity (a.u.)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Fitting 2 peaks...
✅ CSV saved to: 240326TE_S2_CL_5kV_1p6nA_1200_8K_004_PeakSearch_032606_162416.csv
Displaying Window 1: Model Fit...
Displaying Window 2: Literature Map...
